In [1]:
import pandas as pd
import numpy as np

In [2]:
# ============================================
# WEEK 2 - COHORT RETENTION MATRIX
# Owner: Beckley
# ============================================

# Load the dataset
df = pd.read_csv('data/OnlineRetail.csv', encoding='latin1')

# Quick confirmation
print(df.shape)
print(df.columns.tolist())

(541909, 8)
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [3]:
# Apply cleaning steps
df = df.dropna(subset=['CustomerID'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Extract InvoiceMonth
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# Calculate CohortMonth - first purchase month per customer
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')

print(df.shape)
print(df[['CustomerID', 'InvoiceMonth', 'CohortMonth']].head(10))

(397884, 10)
   CustomerID InvoiceMonth CohortMonth
0     17850.0      2010-12     2010-12
1     17850.0      2010-12     2010-12
2     17850.0      2010-12     2010-12
3     17850.0      2010-12     2010-12
4     17850.0      2010-12     2010-12
5     17850.0      2010-12     2010-12
6     17850.0      2010-12     2010-12
7     17850.0      2010-12     2010-12
8     17850.0      2010-12     2010-12
9     13047.0      2010-12     2010-12


In [4]:
# ============================================
# CALCULATE COHORT INDEX
# ============================================

# Extract year and month separately for both dates
invoice_year = df['InvoiceMonth'].dt.year
invoice_month = df['InvoiceMonth'].dt.month

cohort_year = df['CohortMonth'].dt.year
cohort_month = df['CohortMonth'].dt.month

# CohortIndex = number of months between InvoiceMonth and CohortMonth
df['CohortIndex'] = (invoice_year - cohort_year) * 12 + (invoice_month - cohort_month) + 1

print(df[['CustomerID', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']].head(10))

   CustomerID InvoiceMonth CohortMonth  CohortIndex
0     17850.0      2010-12     2010-12            1
1     17850.0      2010-12     2010-12            1
2     17850.0      2010-12     2010-12            1
3     17850.0      2010-12     2010-12            1
4     17850.0      2010-12     2010-12            1
5     17850.0      2010-12     2010-12            1
6     17850.0      2010-12     2010-12            1
7     17850.0      2010-12     2010-12            1
8     17850.0      2010-12     2010-12            1
9     13047.0      2010-12     2010-12            1


In [5]:
# ============================================
# BUILD COHORT RETENTION MATRIX
# ============================================

cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()

cohort_matrix = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')

print(cohort_matrix)

CohortIndex     1      2      3      4      5      6      7      8      9   \
CohortMonth                                                                  
2010-12      885.0  324.0  286.0  340.0  321.0  352.0  321.0  309.0  313.0   
2011-01      417.0   92.0  111.0   96.0  134.0  120.0  103.0  101.0  125.0   
2011-02      380.0   71.0   71.0  108.0  103.0   94.0   96.0  106.0   94.0   
2011-03      452.0   68.0  114.0   90.0  101.0   76.0  121.0  104.0  126.0   
2011-04      300.0   64.0   61.0   63.0   59.0   68.0   65.0   78.0   22.0   
2011-05      284.0   54.0   49.0   49.0   59.0   66.0   75.0   27.0    NaN   
2011-06      242.0   42.0   38.0   64.0   56.0   81.0   23.0    NaN    NaN   
2011-07      188.0   34.0   39.0   42.0   51.0   21.0    NaN    NaN    NaN   
2011-08      169.0   35.0   42.0   41.0   21.0    NaN    NaN    NaN    NaN   
2011-09      299.0   70.0   90.0   34.0    NaN    NaN    NaN    NaN    NaN   
2011-10      358.0   86.0   41.0    NaN    NaN    NaN    NaN    

In [6]:
# ============================================
# CONVERT TO RETENTION PERCENTAGES
# ============================================

# Get the size of each cohort (CohortIndex 1 = starting size)
cohort_sizes = cohort_matrix[1]

# Divide every column by the cohort size, then multiply by 100
retention_matrix = cohort_matrix.divide(cohort_sizes, axis=0) * 100

print(retention_matrix.round(1))

CohortIndex     1     2     3     4     5     6     7     8     9     10  \
CohortMonth                                                                
2010-12      100.0  36.6  32.3  38.4  36.3  39.8  36.3  34.9  35.4  39.5   
2011-01      100.0  22.1  26.6  23.0  32.1  28.8  24.7  24.2  30.0  32.6   
2011-02      100.0  18.7  18.7  28.4  27.1  24.7  25.3  27.9  24.7  30.5   
2011-03      100.0  15.0  25.2  19.9  22.3  16.8  26.8  23.0  27.9   8.6   
2011-04      100.0  21.3  20.3  21.0  19.7  22.7  21.7  26.0   7.3   NaN   
2011-05      100.0  19.0  17.3  17.3  20.8  23.2  26.4   9.5   NaN   NaN   
2011-06      100.0  17.4  15.7  26.4  23.1  33.5   9.5   NaN   NaN   NaN   
2011-07      100.0  18.1  20.7  22.3  27.1  11.2   NaN   NaN   NaN   NaN   
2011-08      100.0  20.7  24.9  24.3  12.4   NaN   NaN   NaN   NaN   NaN   
2011-09      100.0  23.4  30.1  11.4   NaN   NaN   NaN   NaN   NaN   NaN   
2011-10      100.0  24.0  11.5   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
2011-11     